# 📱 Notebook 2: Monolith API vs. BFF — measured

We simulate two downstream services with artificial latency and compare total bytes + time for mobile/web clients.

## 🛠️ Setup

```bash
cd 05-microservices/bff
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
import time, json, random

def slow(ms):
    time.sleep(ms/1000)

def svc_user(uid):
    slow(30)
    return {'id': uid, 'name': 'Ada', 'email': 'a@x.io', 'bio': 'x'*200}

def svc_orders(uid):
    slow(40)
    return [{'id': i, 'total': round(random.random()*100,2), 'lines': ['x']*5} for i in range(5)]

def svc_recs(uid):
    slow(60)
    return [{'sku': f's{i}', 'score': random.random()} for i in range(20)]


## Universal API — client calls all three itself

In [ ]:
def universal_mobile_flow(uid):
    t0 = time.time()
    payload = {'u': svc_user(uid), 'o': svc_orders(uid), 'r': svc_recs(uid)}
    return time.time()-t0, len(json.dumps(payload))

t, b = universal_mobile_flow(1)
print(f'universal: {t*1000:.0f} ms, {b} bytes shipped to mobile')


## Mobile BFF — shapes the response server-side

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def mobile_bff(uid):
    t0 = time.time()
    with ThreadPoolExecutor() as ex:
        f_u = ex.submit(svc_user, uid)
        f_o = ex.submit(svc_orders, uid)
    u, o = f_u.result(), f_o.result()
    payload = {'name': u['name'], 'order_count': len(o)}
    return time.time()-t0, len(json.dumps(payload))

t, b = mobile_bff(1)
print(f'mobile BFF: {t*1000:.0f} ms, {b} bytes shipped to mobile')


### Takeaways
- BFF **parallelised** downstream fan-out (user+orders in parallel) → lower latency.
- BFF **dropped** the recommendation call entirely (mobile doesn't need it).
- Bytes on the wire shrink by ~100x.

> ⚠️ Trade-off: more gateways to operate. Only add a BFF when a client's needs really diverge.